In [74]:
%pip install natsort
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from IPython.display import display
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import re
from natsort import natsorted

# =============================================================================
# 1. 参数区
# =============================================================================
folder_path = Path(r"D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007")
SOH_FILENAME_PATTERN = re.compile(r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$", flags=re.IGNORECASE)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(filename.strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))


SOC_ORDER = ["90%", "50%", "10%"]

CYCLE_ACTIVE_HOUR = 3.0
PAUSE_AFTER_CYCLE_HOUR = 4.5

# 相邻 cycle 起点之间的总间隔
CYCLE_PERIOD_HOUR = CYCLE_ACTIVE_HOUR + PAUSE_AFTER_CYCLE_HOUR
CYCLE_PERIOD_MIN = CYCLE_PERIOD_HOUR * 60

CURRENT_ROUND_DIGITS = 1

OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current_level",
    "Zustand",
    "ID"
]

Note: you may need to restart the kernel to use updated packages.


In [75]:
# =============================================================================
# 2. 读取 parquet
# =============================================================================

parquet_files = natsorted(list(folder_path.rglob("*.parquet")))

if len(parquet_files) == 0:
    raise FileNotFoundError(f"没有在文件夹中找到 parquet 文件: {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name
    temp["SOH"] = extract_soh_from_filename(file.name)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [76]:
# =============================================================================
# 3. 按时间划分 SOC
# =============================================================================

df["Time"] = pd.to_datetime(df["Time"], utc=True, errors="coerce")
df["Current"] = pd.to_numeric(df["Current"], errors="coerce")

df = df.dropna(subset=["Time", "Current"]).copy()
df = df.sort_values("Time").reset_index(drop=True)

# 第一行就是 cycle 1 起点，SOC = 90%
df = df.sort_values(["File", "Time"]).reset_index(drop=True)

file_start_time = df.groupby("File")["Time"].transform("min")

df["elapsed_min"] = (
    df["Time"] - file_start_time
).dt.total_seconds() / 60

df["cycle_id"] = (df["elapsed_min"] // CYCLE_PERIOD_MIN).astype(int) + 1

df["SOC"] = df["cycle_id"].apply(
    lambda x: SOC_ORDER[(x - 1) % len(SOC_ORDER)]
)

df["Current_level"] = df["Current"].round(CURRENT_ROUND_DIGITS)

In [77]:
# =============================================================================
# 4. 只保留读取期间的脉冲
# =============================================================================

df["Zustand"] = df["Zustand"].astype(str)
df["Zustand_raw"] = df["Zustand"].astype(str)
df["Zustand"] = df["Zustand_raw"]
df["Zustand_upper"] = df["Zustand_raw"].str.upper()
df.loc[df["Zustand_upper"].str.startswith("DCH", na=False), "Zustand"] = "DCH"
df.loc[df["Zustand_upper"].str.startswith("CHA", na=False), "Zustand"] = "CHA"
df["Zustand_upper"] = df["Zustand"].str.upper()

df["pulse_segment_id"] = (
    df["File"].ne(df["File"].shift())
    | df["Zustand_upper"].ne(df["Zustand_upper"].shift())
).cumsum()

pulse_mask = df["Zustand_upper"].str.startswith(("CHA", "DCH"), na=False)
drop_dch_1p5_mask = (
    df["Zustand_upper"].str.startswith("DCH", na=False)
    & df["Current_level"].abs().eq(1.5)
)

pulse_sequence = df[pulse_mask & ~drop_dch_1p5_mask].copy()
pulse_sequence = pulse_sequence.sort_values(["pulse_segment_id", "Time"])
pulse_sequence = pulse_sequence.groupby(["File", "pulse_segment_id"], as_index=False).head(1)
pulse_sequence = pulse_sequence.reset_index(drop=True)

pulse_sequence = pulse_sequence[OUTPUT_COLUMNS].copy()

display(pulse_sequence)

,SOH,SOC,File,Time,Current_level,Zustand,ID
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.510000+00:00,1.5,CHA,12_21
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.700000+00:00,-3.0,DCH,12_25
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 15:06:22.210000+00:00,3.0,CHA,12_29
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.720000+00:00,1.5,CHA,12_39
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.020000+00:00,-3.0,DCH,12_43
...,...,...,...,...,...,...,...
215,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.320000+00:00,-3.0,DCH,9_44
216,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.610000+00:00,3.0,CHA,9_48
217,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:31.800000+00:00,1.5,CHA,9_58
218,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.960000+00:00,-3.0,DCH,9_62


In [78]:
# =============================================================================
# 5. 保存结果
# =============================================================================

output_folder = folder_path / "cycle_time_output"
output_folder.mkdir(exist_ok=True)

output_file = output_folder / "recognition_sequence_clean.csv"

pulse_sequence.to_csv(output_file, index=False, encoding="utf-8-sig")

print("保存完成:", output_file)

保存完成: D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007\cycle_time_output\recognition_sequence_clean.csv
